In [1]:
import numpy as np
import scipy
import sklearn
import torch
import transformers

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

assert np.__version__.startswith("2.0"), "NumPy should be 2.0.x here"

numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1
torch: 2.10.0+cu128
transformers: 5.0.0


In [2]:
# ==========================================================
# RAM-optimized LoRA training
# - disk memmap tokenization
# - chunked parquet processing
# - short truncated item texts
# - 2 GPU via DataParallel
# ==========================================================

import os
import sys
import gc
import math
import shutil
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_scheduler,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score


# =========================
# CONFIG
# =========================

ITEMS_PATH = "/kaggle/input/datasets/maxamsecret/ozon-ecup-2026-matching/items.parquet"
MATCHES_PATH = "/kaggle/input/datasets/maxamsecret/ozon-ecup-2026-matching/matches.parquet"

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-12-v2"
OUTPUT_DIR = "./lora_mini_self_attention_memmap"
CACHE_DIR = "./token_cache_memmap"

EPOCHS = 10
SEED = 42

# Важно: 128 сильно экономит RAM/VRAM.
# Если RAM/диск позволяют и нужно качество, можно поставить 256.
MAX_LENGTH = 128

# Обрезаем тексты, чтобы не тащить гигантские атрибуты в RAM.
MAX_NAME_CHARS = 128
MAX_CATEGORY_CHARS = 64
MAX_ATTR_CHARS = 384
MAX_ITEM_CHARS = MAX_NAME_CHARS + MAX_CATEGORY_CHARS + MAX_ATTR_CHARS + 64

# Чанки для токенизации. Если всё ещё OOM RAM -> уменьшай до 2000/1000.
CHUNK_SIZE = 5000
TOKENIZE_BATCH = 5000

# Effective optimizer batch size = 2048
TOTAL_BATCH_SIZE = 2048

# Micro-batch через обе T4. Если OOM VRAM -> 256/128.
MICRO_BATCH_SIZE = 512

EVAL_BATCH_SIZE = 1024

LR = 2e-4
WEIGHT_DECAY = 0.01

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.10

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["OMP_NUM_THREADS"] = str(os.cpu_count() or 8)
os.environ["MKL_NUM_THREADS"] = str(os.cpu_count() or 8)

if TOTAL_BATCH_SIZE % MICRO_BATCH_SIZE != 0:
    raise ValueError("TOTAL_BATCH_SIZE must be divisible by MICRO_BATCH_SIZE")

GRADIENT_ACCUMULATION_STEPS = TOTAL_BATCH_SIZE // MICRO_BATCH_SIZE

set_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Очистим старый кэш токенизации, чтобы не было битых файлов
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)

os.makedirs(CACHE_DIR, exist_ok=True)

print("=" * 90)
print("CONFIG")
print("=" * 90)
print(f"model:                       {MODEL_NAME}")
print(f"max_length:                  {MAX_LENGTH}")
print(f"chunk_size:                  {CHUNK_SIZE}")
print(f"tokenize_batch:              {TOKENIZE_BATCH}")
print(f"total_batch_size:            {TOTAL_BATCH_SIZE}")
print(f"micro_batch_size:            {MICRO_BATCH_SIZE}")
print(f"gradient_accumulation_steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"cache_dir:                   {CACHE_DIR}")
print(f"output_dir:                  {OUTPUT_DIR}")
print("=" * 90)


# =========================
# UTILS
# =========================

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(np.unique(y_true)) < 2:
        return float("nan"), float("nan")

    pos_ap = average_precision_score(y_true, y_prob)
    neg_ap = average_precision_score(1 - y_true, 1 - y_prob)

    macro_pr_auc = float((pos_ap + neg_ap) / 2.0)
    pos_pr_auc = float(pos_ap)

    return macro_pr_auc, pos_pr_auc


# =========================
# LOAD MATCHES
# =========================

print("[DATA] Loading matches.parquet ...")

matches_pl = pl.read_parquet(MATCHES_PATH)

if "target" not in matches_pl.columns:
    raise ValueError("matches.parquet must contain column 'target'")

matches_pl = (
    matches_pl
    .filter(pl.col("id1").is_not_null() & pl.col("id2").is_not_null())
    .with_columns([
        pl.col("target").fill_null(0).cast(pl.Float32),
        (pl.col("target") >= 0.5).cast(pl.Int8).alias("label"),
    ])
    .select(["id1", "id2", "label"])
)

# matches относительно небольшие, можно в pandas для удобного split
matches_pd = matches_pl.to_pandas()

del matches_pl
gc.collect()

print(f"[DATA] matches rows: {len(matches_pd):,}")

labels_all = matches_pd["label"].to_numpy().astype(np.int64)
indices = np.arange(len(labels_all), dtype=np.int64)

try:
    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED,
        stratify=labels_all
    )
except Exception:
    train_idx, val_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED
    )

train_pd = matches_pd.iloc[train_idx].reset_index(drop=True)
val_pd = matches_pd.iloc[val_idx].reset_index(drop=True)

print(f"[DATA] train rows: {len(train_pd):,}")
print(f"[DATA] val rows:   {len(val_pd):,}")

needed_ids = pd.unique(
    pd.concat([
        matches_pd["id1"],
        matches_pd["id2"]
    ])
).tolist()

del matches_pd, labels_all, indices, train_idx, val_idx
gc.collect()

print(f"[DATA] unique item ids needed: {len(needed_ids):,}")


# =========================
# LOAD ITEM TEXTS (TRUNCATED)
# =========================

print("[DATA] Loading items.parquet only for needed ids ...")

items = (
    pl.scan_parquet(ITEMS_PATH)
    .filter(pl.col("id").is_in(needed_ids))
    .select(["id", "name", "category", "attributes"])
    .collect()
)

print(f"[DATA] items loaded: {items.height:,}")

items = items.with_columns([
    pl.col("name")
      .fill_null("")
      .cast(pl.Utf8)
      .str.slice(0, MAX_NAME_CHARS),

    pl.col("category")
      .fill_null("")
      .cast(pl.Utf8)
      .str.slice(0, MAX_CATEGORY_CHARS),

    pl.col("attributes")
      .fill_null("")
      .cast(pl.Utf8, strict=False)
      .str.slice(0, MAX_ATTR_CHARS),
])

# Максимально простой и RAM-эффективный текст без тяжёлого JSON parse
items = items.with_columns(
    (
        pl.lit("Name: ") + pl.col("name") +
        pl.lit(" Category: ") + pl.col("category") +
        pl.lit(" Attributes: ") + pl.col("attributes")
    )
    .str.slice(0, MAX_ITEM_CHARS)
    .alias("text")
)

item_texts = items.select(["id", "text"])

del items, needed_ids
gc.collect()

print(f"[DATA] item_texts rows: {item_texts.height:,}")


# =========================
# TOKENIZER
# =========================

print("[TOKENIZER] Loading tokenizer ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# =========================
# MEMMAP TOKENIZATION
# =========================

def tokenize_to_memmap(
    pairs_pd: pd.DataFrame,
    item_texts: pl.DataFrame,
    tokenizer,
    prefix: str,
    max_length: int,
    chunk_size: int = 5000,
    tokenize_batch: int = 5000
):
    n_max = len(pairs_pd)

    input_ids_path = os.path.join(CACHE_DIR, f"{prefix}_input_ids.bin")
    attention_path = os.path.join(CACHE_DIR, f"{prefix}_attention_mask.bin")
    token_type_path = os.path.join(CACHE_DIR, f"{prefix}_token_type_ids.bin")
    labels_path = os.path.join(CACHE_DIR, f"{prefix}_labels.bin")

    # удаляем старые файлы
    for p in [input_ids_path, attention_path, token_type_path, labels_path]:
        if os.path.exists(p):
            os.remove(p)

    input_ids_mm = np.memmap(
        input_ids_path,
        dtype=np.int32,
        mode="w+",
        shape=(n_max, max_length)
    )

    attention_mm = np.memmap(
        attention_path,
        dtype=np.int8,
        mode="w+",
        shape=(n_max, max_length)
    )

    token_type_mm = np.memmap(
        token_type_path,
        dtype=np.int8,
        mode="w+",
        shape=(n_max, max_length)
    )

    labels_mm = np.memmap(
        labels_path,
        dtype=np.int8,
        mode="w+",
        shape=(n_max,)
    )

    item_texts_id1 = item_texts.rename({"id": "id1", "text": "text1"})
    item_texts_id2 = item_texts.rename({"id": "id2", "text": "text2"})

    written = 0

    total_chunks = math.ceil(n_max / chunk_size)

    for start in tqdm(
        range(0, n_max, chunk_size),
        total=total_chunks,
        desc=f"Tokenize {prefix}",
        dynamic_ncols=True
    ):
        chunk_pd = pairs_pd.iloc[start:start + chunk_size]

        chunk_pl = pl.from_pandas(chunk_pd)

        joined = (
            chunk_pl
            .join(item_texts_id1, on="id1", how="inner")
            .join(item_texts_id2, on="id2", how="inner")
        )

        if joined.height == 0:
            continue

        texts1 = joined.get_column("text1").to_list()
        texts2 = joined.get_column("text2").to_list()
        labels = joined.get_column("label").to_numpy().astype(np.int8)

        for t_start in range(0, joined.height, tokenize_batch):
            t_end = min(t_start + tokenize_batch, joined.height)

            enc = tokenizer(
                text=texts1[t_start:t_end],
                text_pair=texts2[t_start:t_end],
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_tensors="np"
            )

            b = enc["input_ids"].shape[0]

            if written + b > n_max:
                # На практике не должно происходить
                b = n_max - written

            input_ids_mm[written:written + b] = enc["input_ids"].astype(np.int32)
            attention_mm[written:written + b] = enc["attention_mask"].astype(np.int8)

            if "token_type_ids" in enc:
                token_type_mm[written:written + b] = enc["token_type_ids"].astype(np.int8)
            else:
                token_type_mm[written:written + b] = 0

            labels_mm[written:written + b] = labels[t_start:t_end]

            written += b

            if written >= n_max:
                break

        del joined, texts1, texts2, labels, chunk_pl, chunk_pd
        gc.collect()

    input_ids_mm.flush()
    attention_mm.flush()
    token_type_mm.flush()
    labels_mm.flush()

    del input_ids_mm, attention_mm, token_type_mm, labels_mm
    gc.collect()

    paths = {
        "input_ids": input_ids_path,
        "attention_mask": attention_path,
        "token_type_ids": token_type_path,
        "labels": labels_path,
    }

    return paths, written


print("[TOKENIZE] Building train memmap ...")

train_paths, train_len = tokenize_to_memmap(
    pairs_pd=train_pd,
    item_texts=item_texts,
    tokenizer=tokenizer,
    prefix="train",
    max_length=MAX_LENGTH,
    chunk_size=CHUNK_SIZE,
    tokenize_batch=TOKENIZE_BATCH
)

print(f"[TOKENIZE] train effective rows: {train_len:,}")

print("[TOKENIZE] Building val memmap ...")

val_paths, val_len = tokenize_to_memmap(
    pairs_pd=val_pd,
    item_texts=item_texts,
    tokenizer=tokenizer,
    prefix="val",
    max_length=MAX_LENGTH,
    chunk_size=CHUNK_SIZE,
    tokenize_batch=TOKENIZE_BATCH
)

print(f"[TOKENIZE] val effective rows: {val_len:,}")

if train_len == 0 or val_len == 0:
    raise RuntimeError("Train or val dataset is empty after filtering.")

# Теперь тексты и пары больше не нужны
del item_texts, train_pd, val_pd
gc.collect()


# =========================
# MEMMAP DATASET / LOADER
# =========================

class IndexDataset(Dataset):
    def __init__(self, length: int):
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return idx


def make_memmap_loader(paths, length, max_length, batch_size, shuffle):
    if length <= 0:
        raise ValueError("length must be > 0")

    input_ids_mm = np.memmap(
        paths["input_ids"],
        dtype=np.int32,
        mode="r",
        shape=(length, max_length)
    )

    attention_mm = np.memmap(
        paths["attention_mask"],
        dtype=np.int8,
        mode="r",
        shape=(length, max_length)
    )

    token_type_mm = np.memmap(
        paths["token_type_ids"],
        dtype=np.int8,
        mode="r",
        shape=(length, max_length)
    )

    labels_mm = np.memmap(
        paths["labels"],
        dtype=np.int8,
        mode="r",
        shape=(length,)
    )

    dataset = IndexDataset(length)

    def collate_fn(idx_list):
        idx = np.asarray(idx_list, dtype=np.int64)

        input_ids = torch.from_numpy(
            np.asarray(input_ids_mm[idx], dtype=np.int64)
        )

        attention_mask = torch.from_numpy(
            np.asarray(attention_mm[idx], dtype=np.int64)
        )

        token_type_ids = torch.from_numpy(
            np.asarray(token_type_mm[idx], dtype=np.int64)
        )

        labels = torch.from_numpy(
            np.asarray(labels_mm[idx], dtype=np.float32)
        )

        return input_ids, attention_mask, token_type_ids, labels

    drop_last = shuffle and (length >= batch_size)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=collate_fn,
        pin_memory=torch.cuda.is_available(),
        drop_last=drop_last
    )

    handles = [input_ids_mm, attention_mm, token_type_mm, labels_mm]

    return loader, handles


train_loader, train_handles = make_memmap_loader(
    paths=train_paths,
    length=train_len,
    max_length=MAX_LENGTH,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True
)

val_loader, val_handles = make_memmap_loader(
    paths=val_paths,
    length=val_len,
    max_length=MAX_LENGTH,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False
)


# pos_weight for BCE
train_labels_mm = np.memmap(
    train_paths["labels"],
    dtype=np.int8,
    mode="r",
    shape=(train_len,)
)

pos_rate = float(np.mean(train_labels_mm[:]))

del train_labels_mm
gc.collect()

print(f"[DATA] train pos rate after filtering: {pos_rate:.4f}")


# =========================
# MODEL
# =========================

class CrossEncoderClassifier(nn.Module):
    def __init__(self, encoder, hidden_dim):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        cls_emb = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_emb).view(-1)

        return logits


print("[MODEL] Loading base encoder ...")

try:
    base_model = AutoModel.from_pretrained(
        MODEL_NAME,
        attn_implementation="sdpa"
    )
except TypeError:
    base_model = AutoModel.from_pretrained(MODEL_NAME)

hidden_dim = base_model.config.hidden_size

print("[MODEL] Applying LoRA only on self-attention query/key/value ...")

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["query", "key", "value"]
)


CONFIG
model:                       cross-encoder/ms-marco-MiniLM-L-12-v2
max_length:                  128
chunk_size:                  5000
tokenize_batch:              5000
total_batch_size:            2048
micro_batch_size:            512
gradient_accumulation_steps: 4
cache_dir:                   ./token_cache_memmap
output_dir:                  ./lora_mini_self_attention_memmap
[DATA] Loading matches.parquet ...
[DATA] matches rows: 365,654
[DATA] train rows: 292,523
[DATA] val rows:   73,131
[DATA] unique item ids needed: 711,304
[DATA] Loading items.parquet only for needed ids ...


KeyboardInterrupt: 

In [3]:

# FIX: отключаем torchao внутри PEFT, чтобы обойти несовместимую версию
try:
    import peft.import_utils
    import peft.tuners.lora.torchao

    peft.import_utils.is_torchao_available = lambda: False
    peft.tuners.lora.torchao.is_torchao_available = lambda: False

    print("✅ torchao disabled for PEFT")
except Exception as e:
    print(f"⚠️ patch failed: {e}")

peft_encoder = get_peft_model(base_model, lora_config)

model = CrossEncoderClassifier(
    encoder=peft_encoder,
    hidden_dim=hidden_dim
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0

if N_GPU > 1:
    print(f"[GPU] Using DataParallel on {N_GPU} GPUs")
    model = nn.DataParallel(model, device_ids=list(range(N_GPU)))
else:
    print("[GPU] Using single device")

trainable_params = [p for p in model.parameters() if p.requires_grad]

total_params = sum(p.numel() for p in model.parameters())
trainable_params_count = sum(p.numel() for p in trainable_params)

print(f"[MODEL] Total params:     {total_params:,}")
print(f"[MODEL] Trainable params: {trainable_params_count:,}")
print(f"[MODEL] Trainable share:  {100.0 * trainable_params_count / max(1, total_params):.4f}%")


# =========================
# OPTIMIZER / SCHEDULER
# =========================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_training_steps = num_update_steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * 0.05)

scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print(f"[TRAIN] steps per epoch: {num_update_steps_per_epoch}")
print(f"[TRAIN] total steps:     {total_training_steps}")
print(f"[TRAIN] warmup steps:    {warmup_steps}")


# =========================
# LOSS
# =========================

if pos_rate <= 0.0 or pos_rate >= 1.0:
    pos_weight_value = 1.0
else:
    pos_weight_value = (1.0 - pos_rate) / pos_rate
    pos_weight_value = float(np.clip(pos_weight_value, 0.1, 20.0))

print(f"[LOSS] BCE pos_weight: {pos_weight_value:.4f}")

pos_weight = torch.tensor([pos_weight_value], device=device, dtype=torch.float32)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


# =========================
# EVAL FUNCTION
# =========================

def evaluate(eval_model, val_loader, loss_fn, device):
    eval_model.eval()

    all_logits = []
    all_labels = []

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Eval", dynamic_ncols=True, leave=False):
            input_ids, attention_mask, token_type_ids, batch_labels = batch

            input_ids = input_ids.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            token_type_ids = token_type_ids.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)

            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = eval_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids
                    )
            else:
                logits = eval_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )

            loss = loss_fn(logits, batch_labels.float())

            total_loss += loss.item()
            n_batches += 1

            all_logits.append(logits.float().cpu())
            all_labels.append(batch_labels.cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)

    val_loss = total_loss / max(1, n_batches)

    probs = torch.sigmoid(logits).cpu().numpy()
    y_true = labels.long().cpu().numpy()

    macro_pr_auc, pos_pr_auc = compute_metrics(y_true, probs)

    del logits, labels, probs, y_true, all_logits, all_labels
    gc.collect()

    return {
        "val_loss": float(val_loss),
        "macro_pr_auc": float(macro_pr_auc),
        "pos_pr_auc": float(pos_pr_auc)
    }


# =========================
# TRAIN LOOP
# =========================

log_rows = []
best_macro = -1.0

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    ema_loss = 0.0
    first_loss = True

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        dynamic_ncols=True
    )

    for step, batch in enumerate(pbar):
        input_ids, attention_mask, token_type_ids, batch_labels = batch

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        token_type_ids = token_type_ids.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
                loss = loss_fn(logits, batch_labels.float())
        else:
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            loss = loss_fn(logits, batch_labels.float())

        loss_item = loss.item()
        loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if first_loss:
            ema_loss = loss_item
            first_loss = False
        else:
            ema_loss = 0.98 * ema_loss + 0.02 * loss_item

        need_step = (
            ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0) or
            ((step + 1) == len(train_loader))
        )

        if need_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()
            scheduler.step()

        pbar.set_postfix(
            loss=round(ema_loss, 5),
            lr=round(scheduler.get_last_lr()[0], 8)
        )

    train_loss = ema_loss

    print(f"[EPOCH {epoch + 1}] train_loss={train_loss:.5f} | starting validation ...")

    # Для evaluation не используем DataParallel, чтобы не ловить ошибку на последнем маленьком батче
    eval_model = model.module if isinstance(model, nn.DataParallel) else model

    metrics = evaluate(eval_model, val_loader, loss_fn, device)

    row = {
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": metrics["val_loss"],
        "macro_pr_auc": metrics["macro_pr_auc"],
        "pos_pr_auc": metrics["pos_pr_auc"]
    }

    log_rows.append(row)

    pl.DataFrame(log_rows).write_csv(os.path.join(OUTPUT_DIR, "train_log.csv"))

    print("=" * 90)
    print(
        f"[RESULT] epoch={epoch + 1:02d} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={metrics['val_loss']:.5f} | "
        f"macro_pr_auc={metrics['macro_pr_auc']:.5f} | "
        f"pos_pr_auc={metrics['pos_pr_auc']:.5f}"
    )
    print("=" * 90)

    if not np.isnan(metrics["macro_pr_auc"]) and metrics["macro_pr_auc"] > best_macro:
        best_macro = metrics["macro_pr_auc"]

        save_dir = os.path.join(OUTPUT_DIR, "best")
        os.makedirs(save_dir, exist_ok=True)

        raw_model = model.module if isinstance(model, nn.DataParallel) else model

        raw_model.encoder.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)

        torch.save(
            raw_model.classifier.state_dict(),
            os.path.join(save_dir, "classifier.pt")
        )

        print(f"[SAVE] New best model saved to {save_dir} | macro_pr_auc={best_macro:.5f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================
# SAVE LAST
# =========================

last_dir = os.path.join(OUTPUT_DIR, "last")
os.makedirs(last_dir, exist_ok=True)

raw_model = model.module if isinstance(model, nn.DataParallel) else model

raw_model.encoder.save_pretrained(last_dir)
tokenizer.save_pretrained(last_dir)

torch.save(
    raw_model.classifier.state_dict(),
    os.path.join(last_dir, "classifier.pt")
)

print(f"[DONE] Best macro_pr_auc={best_macro:.5f}")
print(f"[DONE] Logs saved to {os.path.join(OUTPUT_DIR, 'train_log.csv')}")
print(f"[DONE] Best model dir: {os.path.join(OUTPUT_DIR, 'best')}")
print(f"[DONE] Last model dir: {os.path.join(OUTPUT_DIR, 'last')}")

✅ torchao disabled for PEFT
[GPU] Using DataParallel on 2 GPUs
[MODEL] Total params:     33,802,753
[MODEL] Trainable params: 442,753
[MODEL] Trainable share:  1.3098%
[TRAIN] steps per epoch: 143
[TRAIN] total steps:     1430
[TRAIN] warmup steps:    71
[LOSS] BCE pos_weight: 2.8945


/tmp/ipykernel_145/1767892548.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Epoch 1/10:   0%|          | 0/571 [00:00<?, ?it/s]

[EPOCH 1] train_loss=0.95411 | starting validation ...


Eval:   0%|          | 0/72 [00:00<?, ?it/s]

...

[EPOCH 7] train_loss=0.80145 | starting validation ...


Eval:   0%|          | 0/72 [00:00<?, ?it/s]

[RESULT] epoch=07 | train_loss=0.80145 | val_loss=0.79708 | macro_pr_auc=0.76823 | pos_pr_auc=0.61496
[SAVE] New best model saved to ./lora_mini_self_attention_memmap/best | macro_pr_auc=0.76823


Epoch 8/10:   0%|          | 0/571 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [4]:
# 1. Установить kaggle API если нет
!pip install -q kaggle

# 2. Создать архив нужных файлов
import shutil
import os

# Укажи что хочешь сохранить
source_dir = "/kaggle/working/lora_mini_self_attention_memmap"
archive_path = "/kaggle/working/lora_model_archive"

# Архивируем
shutil.make_archive(
    base_name=archive_path,
    format='zip',
    root_dir=source_dir
)

print(f"✅ Архив создан: {archive_path}.zip")

# 3. Вариант A: Скачать себе и потом загрузить как датасет вручную
from IPython.display import FileLink
FileLink(f'{archive_path}.zip')

# 4. Вариант B: Загрузить напрямую как новый датасет через API
# (нужен kaggle.json в ~/.kaggle/)
import json

# Создаем metadata для датасета
metadata = {
    "id": "maxamsecret/ozon-lora-minilm-finetuned",  # замени на свой username
    "title": "Ozon LoRA MiniLM Fine-tuned",
    "subtitle": "Cross-encoder with LoRA on self-attention",
    "description": "Fine-tuned MiniLM-L12-v2 with LoRA adapters on Ozon matching data",
    "licenses": [{"name": "CC0-1.0"}]
}

# Сохраняем metadata
metadata_path = "/kaggle/working/dataset-metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f)

# Загружаем как новый датасет
!kaggle datasets create -p /kaggle/working/ --dir-mode zip

print("✅ Датасет создан!")

✅ Архив создан: /kaggle/working/lora_model_archive.zip
Starting upload for file lora_mini_self_attention_memmap.zip
100%|██████████████████████████████████████| 1.78M/1.78M [00:00<00:00, 2.99MB/s]
Upload successful: lora_mini_self_attention_memmap.zip (2MB)
Starting upload for file lora_mini_self_attention_notebook.zip
100%|█████████████████████████████████████████| 22.0/22.0 [00:00<00:00, 34.8B/s]
Upload successful: lora_mini_self_attention_notebook.zip (22B)
Starting upload for file .virtual_documents.zip
100%|██████████████████████████████████████| 7.84k/7.84k [00:00<00:00, 12.6kB/s]
Upload successful: .virtual_documents.zip (8KB)
Starting upload for file lora_model_archive.zip
100%|██████████████████████████████████████| 1.78M/1.78M [00:00<00:00, 2.95MB/s]
Upload successful: lora_model_archive.zip (2MB)
Starting upload for file token_cache_memmap.zip
100%|█████████████████████████████████████████| 22.0/22.0 [00:00<00:00, 30.8B/s]
Upload successful: token_cache_memmap.zip (22B)
Your